**Load Dataset**

In [1]:
from google.colab import userdata
import os

os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_TOKEN')

In [2]:
# Create the kaggle folder
os.makedirs('/root/.kaggle', exist_ok=True)

# Save your token to the access_token file
token = 'KGAT_xxxxxxxxxxxxxxxxxxxx'  # Replace with your token
with open('/root/.kaggle/access_token', 'w') as f:
    f.write(token)

# Set correct permissions
os.chmod('/root/.kaggle/access_token', 0o600)

# Install & download
!pip install kaggle -q
!kaggle datasets download -d kazanova/sentiment140

# Unzip
import zipfile
with zipfile.ZipFile('sentiment140.zip', 'r') as zip_ref:
    zip_ref.extractall('.')

print("✅ Done!")

Dataset URL: https://www.kaggle.com/datasets/kazanova/sentiment140
License(s): other
sentiment140.zip: Skipping, found more recently modified local copy (use --force to force download)
✅ Done!


**Install & Import Libraries**

In [3]:
import numpy as np
import pandas as pd
import re
from  nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import swifter
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [4]:
twitt_data = pd.read_csv("/content/training.1600000.processed.noemoticon.csv",
                          encoding="latin-1",
                          header=None,
                          usecols=[0, 5])
twitt_data.columns = ["target", "text"]

**Clean Dataset**

In [5]:
print(twitt_data.shape)
twitt_data.head()

(1600000, 2)


,target,text
0,0,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,is upset that he can't update his Facebook by ...
2,0,@Kenichan I dived many times for the ball. Man...
3,0,my whole body feels itchy and like its on fire
4,0,"@nationwideclass no, it's not behaving at all...."


In [6]:
twitt_data[['target','text']].isnull().sum()

,0
target,0
text,0


In [7]:
twitt_data['target'].value_counts()

,count
target,
0,800000
4,800000


In [8]:
twitt_data["target"] = twitt_data["target"].replace(4, 1)
twitt_data['target'].value_counts()

,count
target,
0,800000
1,800000


**Text Preprocessing**

In [9]:
stemmer = PorterStemmer()
stop_words = set(stopwords.words("english"))

def stemming(content):
    try:
        stemmed_content = re.sub('[^a-zA-Z]', ' ', content).lower()

        return ' '.join(
            stemmer.stem(word)
            for word in stemmed_content.split()
            if word not in stop_words
        )

    except Exception as e:
        print(f"Error processing content: {content}. Error: {e}")
        return ""

In [10]:
twitt_data['stem_text'] = twitt_data['text'].swifter.apply(stemming)

Pandas Apply:   0%|          | 0/1600000 [00:00<?, ?it/s]

In [11]:
twitt_data.head()


,target,text,stem_text
0,0,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",switchfoot http twitpic com zl awww bummer sho...
1,0,is upset that he can't update his Facebook by ...,upset updat facebook text might cri result sch...
2,0,@Kenichan I dived many times for the ball. Man...,kenichan dive mani time ball manag save rest g...
3,0,my whole body feels itchy and like its on fire,whole bodi feel itchi like fire
4,0,"@nationwideclass no, it's not behaving at all....",nationwideclass behav mad see


**Train/Test Split**

In [12]:
X = twitt_data['text']
y = twitt_data['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 20% test, 80% train
    random_state=42,    # Same split every run
    stratify=y          # Keep 50/50 pos/neg balance in both splits
)

print(f" Train size: {X_train.shape[0]}")
print(f" Test size:  {X_test.shape[0]}")

 Train size: 1280000
 Test size:  320000


**TF-IDF Vectorization**

In [13]:
tfidf = TfidfVectorizer(
    max_features=50000,   # Top 50k words only (keeps it fast)
    ngram_range=(1, 2),   # Single words + two-word pairs ("not good")
    min_df=2              # Ignore words that appear less than twice
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print(f" Train matrix: {X_train_tfidf.shape}")
print(f" Test matrix:  {X_test_tfidf.shape}")

 Train matrix: (1280000, 50000)
 Test matrix:  (320000, 50000)


**Train Logistic Regression**

In [14]:
model = LogisticRegression(
    max_iter=1000,    # Enough iterations to converge
    solver='saga',    # Best solver for large datasets
    n_jobs=-1         # Use all CPU cores
)

model.fit(X_train_tfidf, y_train)

print(" Model trained!")

 Model trained!


In [15]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test_tfidf)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f" Accuracy: {accuracy * 100:.2f}%")

# Full report
print("\n Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))

 Accuracy: 81.98%

 Classification Report:
              precision    recall  f1-score   support

    Negative       0.83      0.81      0.82    160000
    Positive       0.81      0.83      0.82    160000

    accuracy                           0.82    320000
   macro avg       0.82      0.82      0.82    320000
weighted avg       0.82      0.82      0.82    320000



**Test With Custom Tweets**

In [16]:
def predict_sentiment(tweet):
    cleaned = stemming(tweet)
    vectorized = tfidf.transform([cleaned])
    prediction = model.predict(vectorized)[0]
    confidence = model.predict_proba(vectorized)[0]

    label = "Positive " if prediction == 1 else "Negative "
    score = confidence[1] if prediction == 1 else confidence[0]

    print(f"Tweet     : {tweet}")
    print(f"Sentiment : {label}")
    print(f"Confidence: {score * 100:.1f}%")
    print("-" * 50)

In [17]:
predict_sentiment("I can't believe how bad this service is")
predict_sentiment("Today was an amazing day ")

Tweet     : I can't believe how bad this service is
Sentiment : Negative 
Confidence: 99.6%
--------------------------------------------------
Tweet     : Today was an amazing day 
Sentiment : Positive 
Confidence: 65.9%
--------------------------------------------------
